In [1]:
!pip install torchmetrics

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/GMVAE/")

In [8]:
#!pip install -q condacolab
#import condacolab
#condacolab.install()

In [9]:
#!conda install -c conda-forge pyemma -y

In [10]:
import sys
import torch
import numpy as np
import itertools
import os

import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

import pandas as pd
sys.path.append("/content/drive/MyDrive/Colab Notebooks/GMVAE/")
from model import GMVAE
from dataset import load_and_mix_data_nolabel

matplotlib.rcParams.update({'font.size': 24})

In [11]:
!python --version

Python 3.13.15


In [12]:
readdir = os.path.join(os.getcwd(),"data")
modeldir = os.path.join(os.getcwd(),"models")
savedir = os.path.join(os.getcwd(),"results")

In [13]:
data = np.load(os.path.join(readdir,"traj_mb.npy"))
data_list = []
for i in range(data.shape[0]):
    data_list.append(data[i, :, :])
data = data.reshape(len(data)*10000, 2)
dataset = load_and_mix_data_nolabel(data=data, test_ratio=0.01)

In [14]:
dataset

In [15]:
# Check dataset size before training
if hasattr(dataset, 'train'):
    train_size = len(dataset.train.data) if hasattr(dataset.train, 'data') else len(dataset.train)
    test_size = len(dataset.test.data) if hasattr(dataset.test, 'data') else len(dataset.test)

    print(f"Training data size: {train_size}")
    print(f"Test data size: {test_size}")

    # Adjust eval sizes to not exceed dataset size
    n_train_eval_actual = min(200000, train_size)
    n_test_eval_actual = min(200000, test_size)
else:
    n_train_eval_actual = None
    n_test_eval_actual = None

Training data size: 297000
Test data size: 3000


In [16]:
# Training parameters
k, n_x, n_z, n_epochs = 5, 2, 1, 401
qy_dims = [32]
qz_dims = [16]
pz_dims = [16]
px_dims = [128]
r_nent = 0.05
batch_size = 5000
lr = 1e-4

# Setup paths
model_path = modeldir + '/model_' + str(k) + '_' + str(n_z) + '_' + str(n_epochs) + '_' + str(r_nent) + '_' + str(batch_size)
results = '_results'

if not os.path.exists(model_path):
    os.makedirs(model_path)

results_dir_s = model_path + results

if not os.path.exists(results_dir_s):
    os.makedirs(results_dir_s)

results_dir = Path(results_dir_s)

# Initialize model
model = GMVAE(model_path, k=k, n_x=n_x, n_z=n_z, qy_dims=qy_dims,
              qz_dims=qz_dims, pz_dims=pz_dims, px_dims=px_dims,
              r_nent=r_nent, batch_size=batch_size, lr=lr)

In [17]:
# 1. Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Initialize Model
model_path = "./gmvae_model.pt"
k = 10
n_x = 784
n_z = 64
qy_dims = [16, 16]
qz_dims = [16, 16]
px_dims = [16, 16]
pz_dims = [16, 16]
r_nent = 1
batch_size = 1000
lr = 0.0001

model = GMVAE(
    model_path, k=k, n_x=n_x, n_z=n_z,
    qy_dims=qy_dims, qz_dims=qz_dims, pz_dims=pz_dims, px_dims=px_dims,
    r_nent=r_nent, batch_size=batch_size, lr=lr
).to(device)

# 3. Load Data Dataloader (Assuming dataset loader module is available)
from dataset import load_and_mix_data_nolabel
train_loader = load_and_mix_data_nolabel(batch_size=batch_size)

# 4. Invoke Training Loop
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, batch_data in enumerate(train_loader):
        # Extract input tensor if dataset returns tuple (x, y)
        if isinstance(batch_data, (list, tuple)):
            x = batch_data[0]
        else:
            x = batch_data

        x = x.to(device)

        # Invoke training step
        loss_dict = model.train_steps(x)
        running_loss += loss_dict['loss']

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f} | Recon: {loss_dict['recon']:.4f} | Nent: {loss_dict['nent']:.4f}")

# 5. Save Trained Checkpoint
torch.save(model.state_dict(), model.model_path)
print("Training completed and model saved successfully.")

Epoch [1/20] - Loss: 1173.0905 | Recon: 1170.8069 | Nent: 2.2836
Epoch [2/20] - Loss: 1172.8595 | Recon: 1170.5758 | Nent: 2.2836
Epoch [3/20] - Loss: 1172.6630 | Recon: 1170.3795 | Nent: 2.2835
Epoch [4/20] - Loss: 1172.5449 | Recon: 1170.2614 | Nent: 2.2835
Epoch [5/20] - Loss: 1172.4001 | Recon: 1170.1166 | Nent: 2.2835
Epoch [6/20] - Loss: 1172.1444 | Recon: 1169.8610 | Nent: 2.2834
Epoch [7/20] - Loss: 1172.1250 | Recon: 1169.8416 | Nent: 2.2834
Epoch [8/20] - Loss: 1171.9276 | Recon: 1169.6442 | Nent: 2.2833
Epoch [9/20] - Loss: 1171.8235 | Recon: 1169.5402 | Nent: 2.2833
Epoch [10/20] - Loss: 1171.5555 | Recon: 1169.2722 | Nent: 2.2833
Epoch [11/20] - Loss: 1171.4077 | Recon: 1169.1245 | Nent: 2.2832
Epoch [12/20] - Loss: 1171.3794 | Recon: 1169.0962 | Nent: 2.2832
Epoch [13/20] - Loss: 1171.2820 | Recon: 1168.9988 | Nent: 2.2831
Epoch [14/20] - Loss: 1171.0880 | Recon: 1168.8051 | Nent: 2.2831
Epoch [15/20] - Loss: 1170.9385 | Recon: 1168.6555 | Nent: 2.2830
Epoch [16/20] - Los